# CoDA-GQA-L: H100 Paper Experiments

This notebook runs the key experiments needed for the CoDA-GQA-L paper:

1. **Forward Check** - Validate weight transfer on SmolLM2-135M (sanity check)
2. **LoRA Fine-Tuning** - Train CoDA attention on SmolLM2-135M
3. **Perplexity Evaluation** - WikiText-2 PPL: original vs CoDA-unbounded vs CoDA-bounded
4. **Needle-in-Haystack** - Retrieval at depth with bounded configs
5. **Throughput Benchmarks** - H100 numbers for the paper

**Target GPU**: NVIDIA H100 on Runpod

---

## 0. Setup

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets peft accelerate
!pip install -e /workspace/CoDA-QGA-L  # adjust path for your Runpod mount

In [ ]:
import copy
import json
import math
import os
import time
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

from coda_gqa_l import (
    CoDAGQALandmarkPerf2,
    CoDAGQALandmarkStatePerf2,
    CoDAGQA,
    BaselineGQA,
)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Dtype: {dtype}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Results directory
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"

def save_result(data: dict, prefix: str) -> Path:
    """Save experiment results to JSON."""
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = RESULTS_DIR / f"{prefix}_{ts}.json"
    path.write_text(json.dumps(data, indent=2, default=str))
    print(f"Saved: {path}")
    return path

---
## 1. Forward Check (Sanity)

Quick validation that weight transfer works on this GPU.
BaselineGQA swap should give 100% top-1 agreement.

In [ ]:
# Run the generic forward check script
!python ../benchmarks/forward_check_generic.py --model {MODEL_NAME} --seq-len 64

---
## 2. LoRA Fine-Tuning: SmolLM2-135M with CoDA Attention

**Goal**: Prove that CoDA-GQA (differential attention + HeadwiseRMSNorm) can match
the original model's perplexity after light LoRA fine-tuning.

**Strategy**:
1. Swap all attention layers with CoDAGQA (unbounded, for training)
2. Freeze everything except CoDA-specific params (theta, lambda_proj, head_norm)
   and LoRA adapters on q/k/v/o projections
3. Train on WikiText-103 (or a subset)
4. Evaluate on WikiText-2

In [ ]:
# =====================================================================
# Weight transfer utilities (from forward_check_generic.py)
# =====================================================================

def halfsplit_to_interleaved_perm(head_dim: int) -> torch.Tensor:
    """Permutation converting Llama half-split RoPE to interleaved RoPE."""
    half = head_dim // 2
    perm = torch.empty(head_dim, dtype=torch.long)
    for i in range(half):
        perm[2 * i] = i
        perm[2 * i + 1] = half + i
    return perm


def permute_qk_weight(weight: torch.Tensor, num_heads: int, head_dim: int) -> torch.Tensor:
    """Permute Q/K weight output dim for RoPE convention change."""
    perm = halfsplit_to_interleaved_perm(head_dim).to(weight.device)
    embed_dim = weight.shape[1]
    w = weight.view(num_heads, head_dim, embed_dim)
    w = w[:, perm, :]
    return w.reshape(num_heads * head_dim, embed_dim)


class LlamaCoDAWrapper(nn.Module):
    """Wraps CoDAGQA to match Llama's self_attn forward signature."""

    def __init__(self, coda_module: nn.Module):
        super().__init__()
        self.attn = coda_module

    def forward(self, hidden_states, attention_mask=None, position_ids=None,
                past_key_values=None, use_cache=False, cache_position=None,
                position_embeddings=None, **kwargs):
        out, _kv = self.attn(hidden_states, is_causal=True)
        return out, None


def swap_attention_to_coda(model, model_config):
    """Replace all Llama attention layers with CoDAGQA + weight transfer."""
    embed_dim = model_config.hidden_size
    num_heads = model_config.num_attention_heads
    num_kv_heads = getattr(model_config, "num_key_value_heads", num_heads)
    head_dim = embed_dim // num_heads
    rope_theta = getattr(model_config, "rope_theta", 10_000.0)

    n_swapped = 0
    for i, block in enumerate(model.model.layers):
        orig = block.self_attn

        coda = CoDAGQA(
            embed_dim=embed_dim,
            num_heads=num_heads,
            num_kv_heads=num_kv_heads,
            rope_base=rope_theta,
        ).to(device=device, dtype=dtype)

        # Copy weights with RoPE convention permutation
        coda.q_proj.weight.data.copy_(
            permute_qk_weight(orig.q_proj.weight.data, num_heads, head_dim)
        )
        coda.k_proj.weight.data.copy_(
            permute_qk_weight(orig.k_proj.weight.data, num_kv_heads, head_dim)
        )
        coda.v_proj.weight.data.copy_(orig.v_proj.weight.data)
        coda.o_proj.weight.data.copy_(orig.o_proj.weight.data)

        block.self_attn = LlamaCoDAWrapper(coda)
        n_swapped += 1

    print(f"Swapped {n_swapped} attention layers to CoDAGQA")
    print(f"  embed_dim={embed_dim}, heads={num_heads}, kv_heads={num_kv_heads}")
    print(f"  head_dim={head_dim}, rope_theta={rope_theta}")
    return model

In [ ]:
# =====================================================================
# Load model and swap attention
# =====================================================================

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=dtype
).to(device)

print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} params")
print(f"Config: D={model.config.hidden_size}, H={model.config.num_attention_heads}, "
      f"Hkv={model.config.num_key_value_heads}, layers={model.config.num_hidden_layers}")

# Swap attention layers
model = swap_attention_to_coda(model, model.config)

In [ ]:
# =====================================================================
# Configure trainable parameters
# =====================================================================

# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze CoDA-specific parameters in each attention layer
coda_param_names = []
for i, block in enumerate(model.model.layers):
    wrapper = block.self_attn
    coda = wrapper.attn

    # CoDA-unique params: theta (rotation), lambda_proj, head_norm
    for name, param in coda.named_parameters():
        if any(k in name for k in ["theta", "lambda_proj", "head_norm"]):
            param.requires_grad = True
            coda_param_names.append(f"layer.{i}.{name}")

    # Also unfreeze the projection weights for fine-tuning
    # (they have correct weights but need adaptation to CoDA context)
    for name, param in coda.named_parameters():
        if any(k in name for k in ["q_proj", "k_proj", "v_proj", "o_proj"]):
            param.requires_grad = True
            coda_param_names.append(f"layer.{i}.{name}")

# Count trainable params
n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {n_total:,}")
print(f"Trainable params: {n_trainable:,} ({100*n_trainable/n_total:.1f}%)")
print(f"\nTrainable parameter groups: {len(set(coda_param_names))}")

In [ ]:
# =====================================================================
# Prepare dataset: WikiText-103 for training, WikiText-2 for eval
# =====================================================================

SEQ_LEN = 512

def tokenize_and_chunk(dataset, tokenizer, seq_len, max_tokens=None):
    """Tokenize text and chunk into fixed-length sequences."""
    all_ids = []
    for example in dataset:
        text = example["text"]
        if text.strip():
            ids = tokenizer(text, add_special_tokens=False)["input_ids"]
            all_ids.extend(ids)
        if max_tokens and len(all_ids) >= max_tokens:
            all_ids = all_ids[:max_tokens]
            break

    # Chunk into seq_len blocks
    n_chunks = len(all_ids) // seq_len
    all_ids = all_ids[:n_chunks * seq_len]
    chunks = torch.tensor(all_ids, dtype=torch.long).view(n_chunks, seq_len)
    return chunks

print("Loading WikiText datasets...")
wt103 = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")
wt2_test = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")

# Use ~10M tokens for training (fast fine-tune)
MAX_TRAIN_TOKENS = 10_000_000
train_chunks = tokenize_and_chunk(wt103, tokenizer, SEQ_LEN, max_tokens=MAX_TRAIN_TOKENS)
eval_chunks = tokenize_and_chunk(wt2_test, tokenizer, SEQ_LEN)

print(f"Training: {train_chunks.shape[0]} chunks x {SEQ_LEN} = {train_chunks.numel():,} tokens")
print(f"Eval:     {eval_chunks.shape[0]} chunks x {SEQ_LEN} = {eval_chunks.numel():,} tokens")

In [ ]:
# =====================================================================
# Perplexity evaluation function
# =====================================================================

@torch.no_grad()
def evaluate_perplexity(model, eval_data, batch_size=8, desc="Eval"):
    """Compute perplexity on chunked evaluation data."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    loader = DataLoader(eval_data, batch_size=batch_size, shuffle=False)
    for batch_idx, input_ids in enumerate(loader):
        input_ids = input_ids.to(device)
        labels = input_ids[:, 1:].contiguous()
        inputs = input_ids[:, :-1].contiguous()

        outputs = model(inputs)
        if hasattr(outputs, "logits"):
            logits = outputs.logits
        elif isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs

        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            labels.view(-1),
            reduction="sum",
        )
        total_loss += loss.item()
        total_tokens += labels.numel()

        if (batch_idx + 1) % 20 == 0:
            running_ppl = math.exp(total_loss / total_tokens)
            print(f"  {desc}: batch {batch_idx+1}/{len(loader)}, running PPL={running_ppl:.2f}")

    avg_loss = total_loss / total_tokens
    ppl = math.exp(avg_loss)
    print(f"  {desc}: PPL = {ppl:.2f} (loss = {avg_loss:.4f}, tokens = {total_tokens:,})")
    return ppl

In [ ]:
# =====================================================================
# Pre-training perplexity (CoDA cold-swap, expected to be terrible)
# =====================================================================

print("=" * 60)
print("Pre-training perplexity (CoDA cold-swap, no fine-tuning)")
print("=" * 60)
ppl_cold = evaluate_perplexity(model, eval_chunks, batch_size=4, desc="Cold-swap")
print(f"\nCold-swap PPL: {ppl_cold:.2f}  (expected: very high, >1000)")

In [ ]:
# =====================================================================
# Training loop
# =====================================================================

# Hyperparameters
BATCH_SIZE = 4
GRAD_ACCUM = 4         # effective batch = 16
LR = 3e-4
WARMUP_STEPS = 200
MAX_STEPS = 5000       # ~5K steps * 16 * 512 = ~40M tokens
EVAL_EVERY = 500
LOG_EVERY = 50

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.01,
    betas=(0.9, 0.95),
)

# Linear warmup + cosine decay scheduler
def lr_schedule(step):
    if step < WARMUP_STEPS:
        return step / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_schedule)

train_loader = DataLoader(train_chunks, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
train_iter = iter(train_loader)

print(f"Training config:")
print(f"  Batch size: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"  LR: {LR}, Warmup: {WARMUP_STEPS}, Max steps: {MAX_STEPS}")
print(f"  Seq len: {SEQ_LEN}")
print(f"  Tokens per step: {BATCH_SIZE * GRAD_ACCUM * SEQ_LEN:,}")
print(f"  Total tokens: ~{MAX_STEPS * BATCH_SIZE * GRAD_ACCUM * SEQ_LEN / 1e6:.0f}M")

# Training
model.train()
best_ppl = float("inf")
train_log = []
t_start = time.time()

for step in range(1, MAX_STEPS + 1):
    optimizer.zero_grad()
    step_loss = 0.0

    for _ in range(GRAD_ACCUM):
        try:
            input_ids = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            input_ids = next(train_iter)

        input_ids = input_ids.to(device)
        labels = input_ids[:, 1:].contiguous()
        inputs = input_ids[:, :-1].contiguous()

        outputs = model(inputs)
        if hasattr(outputs, "logits"):
            logits = outputs.logits
        elif isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs

        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            labels.view(-1),
        )
        (loss / GRAD_ACCUM).backward()
        step_loss += loss.item() / GRAD_ACCUM

    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], 1.0
    )
    optimizer.step()
    scheduler.step()

    if step % LOG_EVERY == 0:
        elapsed = time.time() - t_start
        tokens_per_sec = step * BATCH_SIZE * GRAD_ACCUM * SEQ_LEN / elapsed
        lr_now = scheduler.get_last_lr()[0]
        ppl_train = math.exp(step_loss)
        print(f"  step {step:5d}/{MAX_STEPS} | loss={step_loss:.4f} | PPL={ppl_train:.2f} | "
              f"lr={lr_now:.2e} | {tokens_per_sec:.0f} tok/s")
        train_log.append({"step": step, "loss": step_loss, "ppl": ppl_train, "lr": lr_now})

    if step % EVAL_EVERY == 0:
        ppl_eval = evaluate_perplexity(model, eval_chunks, batch_size=4, desc=f"Step {step}")
        train_log.append({"step": step, "eval_ppl": ppl_eval})
        if ppl_eval < best_ppl:
            best_ppl = ppl_eval
            # Save best checkpoint
            ckpt_path = RESULTS_DIR / "coda_finetune_best.pt"
            torch.save({
                "step": step,
                "ppl": ppl_eval,
                "model_state": {k: v for k, v in model.state_dict().items()
                                if any(x in k for x in ["theta", "lambda_proj", "head_norm",
                                                         "q_proj", "k_proj", "v_proj", "o_proj"])},
            }, ckpt_path)
            print(f"  New best! PPL={ppl_eval:.2f} -> {ckpt_path}")
        model.train()

total_time = time.time() - t_start
print(f"\nTraining complete: {total_time/60:.1f} min")
print(f"Best eval PPL: {best_ppl:.2f}")

In [ ]:
# =====================================================================
# Final perplexity evaluation
# =====================================================================

print("=" * 60)
print("Final evaluation: CoDA-GQA (unbounded) after fine-tuning")
print("=" * 60)
ppl_coda_unbounded = evaluate_perplexity(model, eval_chunks, batch_size=4, desc="CoDA-unbounded")

# Also evaluate the original model for comparison
print("\nLoading original model for baseline comparison...")
model_orig = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=dtype
).to(device)
model_orig.eval()
ppl_original = evaluate_perplexity(model_orig, eval_chunks, batch_size=4, desc="Original")
del model_orig
torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"PERPLEXITY COMPARISON (WikiText-2):")
print(f"  Original SmolLM2-135M: {ppl_original:.2f}")
print(f"  CoDA cold-swap:        {ppl_cold:.2f}")
print(f"  CoDA fine-tuned:       {ppl_coda_unbounded:.2f}")
print(f"  Delta from original:   {ppl_coda_unbounded - ppl_original:+.2f} ({100*(ppl_coda_unbounded/ppl_original - 1):+.1f}%)")
print(f"{'='*60}")

save_result({
    "experiment": "finetune_perplexity",
    "model": MODEL_NAME,
    "training": {
        "max_steps": MAX_STEPS,
        "batch_size": BATCH_SIZE * GRAD_ACCUM,
        "seq_len": SEQ_LEN,
        "lr": LR,
        "total_time_min": total_time / 60,
    },
    "perplexity": {
        "original": ppl_original,
        "coda_cold_swap": ppl_cold,
        "coda_finetuned_unbounded": ppl_coda_unbounded,
    },
    "train_log": train_log,
    "gpu": torch.cuda.get_device_name(0),
}, "finetune_ppl")

---
## 3. Bounded Perplexity Evaluation

After fine-tuning with unbounded CoDA, evaluate with bounded KV configs.
This tests whether the memory banks preserve quality.

**Note**: Bounded eval requires processing tokens sequentially through `prefill_chunked`
instead of the standard model forward pass.

In [ ]:
# =====================================================================
# Bounded perplexity evaluation
# =====================================================================

# We need to build a bounded model from the fine-tuned weights.
# Strategy: create CoDAGQALandmarkPerf2 modules, copy weights from
# the fine-tuned CoDAGQA modules.

def create_bounded_model_from_finetuned(finetuned_model, model_config, bounded_config):
    """Create a model with bounded CoDA attention from fine-tuned weights.

    Args:
        finetuned_model: Model with CoDAGQA attention (fine-tuned)
        model_config: HuggingFace model config
        bounded_config: dict with window, Me, Ms keys
    """
    embed_dim = model_config.hidden_size
    num_heads = model_config.num_attention_heads
    num_kv_heads = getattr(model_config, "num_key_value_heads", num_heads)
    head_dim = embed_dim // num_heads
    rope_theta = getattr(model_config, "rope_theta", 10_000.0)

    # Deep copy the model (non-attention parts)
    bounded_model = copy.deepcopy(finetuned_model)

    for i, block in enumerate(bounded_model.model.layers):
        coda_unbounded = block.self_attn.attn  # CoDAGQA

        # Create bounded module
        coda_bounded = CoDAGQALandmarkPerf2(
            embed_dim=embed_dim,
            num_heads=num_heads,
            num_kv_heads=num_kv_heads,
            window=bounded_config["window"],
            num_landmarks_exact=bounded_config["Me"],
            num_landmarks_summary=bounded_config["Ms"],
            rope_base=rope_theta,
        ).to(device=device, dtype=dtype)

        # Copy all shared weights
        coda_bounded.q_proj.weight.data.copy_(coda_unbounded.q_proj.weight.data)
        coda_bounded.k_proj.weight.data.copy_(coda_unbounded.k_proj.weight.data)
        coda_bounded.v_proj.weight.data.copy_(coda_unbounded.v_proj.weight.data)
        coda_bounded.o_proj.weight.data.copy_(coda_unbounded.o_proj.weight.data)
        coda_bounded.theta.data.copy_(coda_unbounded.theta.data)
        coda_bounded.lambda_proj.weight.data.copy_(coda_unbounded.lambda_proj.weight.data)
        coda_bounded.lambda_proj.bias.data.copy_(coda_unbounded.lambda_proj.bias.data)
        coda_bounded.head_norm.weight.data.copy_(coda_unbounded.head_norm.weight.data)

        block.self_attn = coda_bounded  # replace directly (no wrapper needed for eval)

    print(f"Created bounded model: W={bounded_config['window']}, "
          f"Me={bounded_config['Me']}, Ms={bounded_config['Ms']}")
    return bounded_model


@torch.no_grad()
def evaluate_bounded_perplexity(bounded_model, eval_data, model_config,
                                 block_size=256, desc="Bounded"):
    """Evaluate perplexity using bounded attention layer by layer.

    This manually runs tokens through the model, using prefill_chunked
    for the CoDA attention layers while keeping MLP/LayerNorm standard.
    """
    # For bounded eval, we need to process sequences through the model
    # using the bounded attention mechanism. The simplest approach:
    # process each eval sequence through the full model, with bounded
    # attention layers managing their own state.
    #
    # Since each CoDAGQALandmarkPerf2 is now the self_attn directly,
    # we need a wrapper that initializes state and routes through prefill.

    class BoundedAttentionHook:
        """Temporary wrapper enabling bounded attention during eval."""
        def __init__(self, coda_module, batch_size):
            self.coda = coda_module
            self.state = coda_module.init_state(
                batch_size=batch_size, device=device, dtype=dtype,
            )
            self.block_size = block_size

        def __call__(self, hidden_states, **kwargs):
            y, self.state = self.coda.prefill_chunked(
                hidden_states, self.state,
                block_size=self.block_size,
                write_cache=True,
                return_outputs=True,
            )
            return y, None

    total_loss = 0.0
    total_tokens = 0

    for seq_idx in range(len(eval_data)):
        input_ids = eval_data[seq_idx:seq_idx+1].to(device)  # (1, seq_len)
        labels = input_ids[:, 1:].contiguous()

        # Set up bounded hooks for this sequence
        hooks = []
        for block in bounded_model.model.layers:
            coda = block.self_attn
            hook = BoundedAttentionHook(coda, batch_size=1)
            # Temporarily replace forward
            original_forward = coda.forward
            block.self_attn = type('BoundedProxy', (), {
                '__call__': lambda self_, hs, **kw: hook(hs, **kw),
                'forward': lambda self_, hs, **kw: hook(hs, **kw),
            })()
            hooks.append((block, original_forward, coda))

        # Run forward
        try:
            outputs = bounded_model(input_ids[:, :-1])
            if hasattr(outputs, "logits"):
                logits = outputs.logits
            elif isinstance(outputs, tuple):
                logits = outputs[0]
            else:
                logits = outputs

            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                labels.view(-1),
                reduction="sum",
            )
            total_loss += loss.item()
            total_tokens += labels.numel()
        finally:
            # Restore original attention modules
            for block, orig_fwd, coda in hooks:
                block.self_attn = coda

        if (seq_idx + 1) % 20 == 0:
            running_ppl = math.exp(total_loss / total_tokens)
            print(f"  {desc}: seq {seq_idx+1}/{len(eval_data)}, running PPL={running_ppl:.2f}")

    avg_loss = total_loss / total_tokens
    ppl = math.exp(avg_loss)
    print(f"  {desc}: PPL = {ppl:.2f}")
    return ppl

In [ ]:
# =====================================================================
# Evaluate bounded configs
# =====================================================================

BOUNDED_CONFIGS = {
    "tiny-cache": {"window": 128, "Me": 32, "Ms": 32},
    "medium-cache": {"window": 256, "Me": 64, "Ms": 64},
    "window-only": {"window": 256, "Me": 0, "Ms": 0},
}

bounded_results = {}

for config_name, config in BOUNDED_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"Evaluating bounded config: {config_name}")
    print(f"  W={config['window']}, Me={config['Me']}, Ms={config['Ms']}")
    print(f"{'='*60}")

    bounded_model = create_bounded_model_from_finetuned(
        model, model.config, config
    )
    bounded_model.eval()

    # Use a subset of eval data (bounded eval is slower due to per-sequence state)
    eval_subset = eval_chunks[:50]  # ~25K tokens
    ppl = evaluate_bounded_perplexity(
        bounded_model, eval_subset, model.config,
        block_size=256, desc=config_name,
    )
    bounded_results[config_name] = ppl

    del bounded_model
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"BOUNDED PERPLEXITY RESULTS:")
print(f"  Original:        {ppl_original:.2f}")
print(f"  CoDA unbounded:  {ppl_coda_unbounded:.2f}")
for name, ppl in bounded_results.items():
    delta = ppl - ppl_coda_unbounded
    print(f"  CoDA {name}: {ppl:.2f} ({delta:+.2f} vs unbounded)")
print(f"{'='*60}")

save_result({
    "experiment": "bounded_perplexity",
    "model": MODEL_NAME,
    "perplexity": {
        "original": ppl_original,
        "coda_unbounded": ppl_coda_unbounded,
        **{f"coda_{k}": v for k, v in bounded_results.items()},
    },
    "configs": BOUNDED_CONFIGS,
    "gpu": torch.cuda.get_device_name(0),
}, "bounded_ppl")

---
## 4. Needle-in-Haystack Retrieval

Tests whether bounded KV configs can retrieve specific tokens
embedded at various depths in a long sequence.

This directly tests Claims 3 (V-routing) and 8 (needle retrieval).

In [ ]:
# =====================================================================
# Needle-in-Haystack test
# =====================================================================

NEEDLE_PASSKEY = "The secret passkey is 83729."
NEEDLE_QUERY = "What is the secret passkey? The secret passkey is"
NEEDLE_ANSWER = " 83729"  # expected continuation


def build_haystack(tokenizer, total_len, needle_pos, needle_text, filler_text=None):
    """Build a haystack sequence with a needle at a specific position.

    Args:
        tokenizer: Tokenizer
        total_len: Total sequence length in tokens
        needle_pos: Token position to insert the needle
        needle_text: The needle text to embed
        filler_text: Filler text (default: repeated sentences)

    Returns:
        input_ids tensor (1, total_len)
    """
    if filler_text is None:
        filler_text = (
            "The weather was pleasant that day with clear skies and a gentle breeze. "
            "Birds sang in the trees while children played in the park nearby. "
            "The sun cast long shadows across the garden path. "
            "Flowers bloomed in every color imaginable along the walkway. "
        )

    needle_ids = tokenizer(needle_text, add_special_tokens=False)["input_ids"]
    filler_ids = tokenizer(filler_text, add_special_tokens=False)["input_ids"]

    # Build sequence: filler up to needle_pos, needle, filler to fill total_len
    result = []
    while len(result) < needle_pos:
        result.extend(filler_ids)
    result = result[:needle_pos]
    result.extend(needle_ids)
    while len(result) < total_len:
        result.extend(filler_ids)
    result = result[:total_len]

    return torch.tensor([result], dtype=torch.long)


@torch.no_grad()
def needle_test_unbounded(model, tokenizer, haystack_ids, query_text):
    """Test needle retrieval with standard (unbounded) forward pass."""
    query_ids = tokenizer(query_text, add_special_tokens=False, return_tensors="pt")["input_ids"]
    full_input = torch.cat([haystack_ids, query_ids.to(haystack_ids.device)], dim=1).to(device)

    outputs = model(full_input)
    logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]

    # Get top-5 predicted tokens after the query
    last_logits = logits[0, -1, :]
    top5 = torch.topk(last_logits, 5)
    top5_tokens = [tokenizer.decode([t]) for t in top5.indices]
    return top5_tokens, top5.values.tolist()


@torch.no_grad()
def needle_test_bounded(coda_module, tokenizer, haystack_ids, query_ids,
                        embed_fn, head_fn, block_size=256):
    """Test needle retrieval with bounded CoDA attention (single layer test).

    For a proper multi-layer test, use the full model with bounded wrappers.
    This is a simplified single-attention-layer probe.
    """
    B, L = haystack_ids.shape

    # Embed the full sequence
    full_ids = torch.cat([haystack_ids, query_ids.to(haystack_ids.device)], dim=1).to(device)
    x = embed_fn(full_ids)  # (1, L_total, D)

    # Process through bounded attention
    state = coda_module.init_state(batch_size=1, device=device, dtype=dtype)
    y, state = coda_module.prefill_chunked(
        x, state, block_size=block_size, write_cache=True, return_outputs=True,
    )

    # Get the output for the last token
    last_hidden = y[:, -1:, :]  # (1, 1, D)
    logits = head_fn(last_hidden)  # (1, 1, vocab)

    top5 = torch.topk(logits[0, 0, :], 5)
    top5_tokens = [tokenizer.decode([t]) for t in top5.indices]
    return top5_tokens, top5.values.tolist()

In [ ]:
# =====================================================================
# Run needle tests at various depths
# =====================================================================

DEPTHS = [64, 128, 256, 512, 1024, 2048]
TOTAL_LEN = 2560  # total haystack length

print("=" * 60)
print("Needle-in-Haystack Retrieval Test")
print(f"Needle: '{NEEDLE_PASSKEY}'")
print(f"Query: '{NEEDLE_QUERY}'")
print(f"Total haystack: {TOTAL_LEN} tokens")
print("=" * 60)

# Load the original model for unbounded comparison
print("\nLoading original model for unbounded needle test...")
model_orig = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=dtype
).to(device)
model_orig.eval()

needle_results = {"depths": DEPTHS, "total_len": TOTAL_LEN, "configs": {}}

# Test with original model (unbounded)
print("\n--- Original (unbounded) ---")
orig_hits = []
for depth in DEPTHS:
    if depth >= TOTAL_LEN:
        orig_hits.append(False)
        continue
    haystack = build_haystack(tokenizer, TOTAL_LEN, depth, NEEDLE_PASSKEY)
    top5, scores = needle_test_unbounded(model_orig, tokenizer, haystack.to(device), NEEDLE_QUERY)
    hit = NEEDLE_ANSWER.strip() in " ".join(top5)
    orig_hits.append(hit)
    status = "HIT" if hit else "MISS"
    print(f"  depth={depth:5d}: {status}  top5={top5}")

needle_results["configs"]["original"] = {
    "hits": orig_hits,
    "accuracy": sum(orig_hits) / len(orig_hits),
}

del model_orig
torch.cuda.empty_cache()

# Test with fine-tuned CoDA (unbounded)
print("\n--- CoDA fine-tuned (unbounded) ---")
model.eval()
coda_unbounded_hits = []
for depth in DEPTHS:
    if depth >= TOTAL_LEN:
        coda_unbounded_hits.append(False)
        continue
    haystack = build_haystack(tokenizer, TOTAL_LEN, depth, NEEDLE_PASSKEY)
    top5, scores = needle_test_unbounded(model, tokenizer, haystack.to(device), NEEDLE_QUERY)
    hit = NEEDLE_ANSWER.strip() in " ".join(top5)
    coda_unbounded_hits.append(hit)
    status = "HIT" if hit else "MISS"
    print(f"  depth={depth:5d}: {status}  top5={top5}")

needle_results["configs"]["coda_unbounded"] = {
    "hits": coda_unbounded_hits,
    "accuracy": sum(coda_unbounded_hits) / len(coda_unbounded_hits),
}

# Test with bounded configs
for config_name, config in BOUNDED_CONFIGS.items():
    print(f"\n--- CoDA bounded: {config_name} ---")
    bounded_model = create_bounded_model_from_finetuned(
        model, model.config, config
    )
    bounded_model.eval()

    bounded_hits = []
    for depth in DEPTHS:
        if depth >= TOTAL_LEN:
            bounded_hits.append(False)
            continue
        haystack = build_haystack(tokenizer, TOTAL_LEN, depth, NEEDLE_PASSKEY)

        # For bounded, use the full model with bounded attention hooks
        # Simple approach: process through model with bounded self_attn modules
        full_input = torch.cat([
            haystack.to(device),
            tokenizer(NEEDLE_QUERY, add_special_tokens=False,
                      return_tensors="pt")["input_ids"].to(device)
        ], dim=1)

        # Initialize bounded state for each layer
        for block in bounded_model.model.layers:
            coda = block.self_attn
            coda._state = coda.init_state(batch_size=1, device=device, dtype=dtype)

        # Process through embeddings
        hidden = bounded_model.model.embed_tokens(full_input)

        # Process through each transformer block with bounded attention
        for block in bounded_model.model.layers:
            residual = hidden
            hidden = block.input_layernorm(hidden)
            coda = block.self_attn
            attn_out, coda._state = coda.prefill_chunked(
                hidden, coda._state,
                block_size=256, write_cache=True, return_outputs=True,
            )
            hidden = residual + attn_out
            # MLP
            residual = hidden
            hidden = block.post_attention_layernorm(hidden)
            hidden = block.mlp(hidden)
            hidden = residual + hidden

        hidden = bounded_model.model.norm(hidden)
        logits = bounded_model.lm_head(hidden)

        top5_ids = torch.topk(logits[0, -1, :], 5).indices
        top5_tokens = [tokenizer.decode([t]) for t in top5_ids]
        hit = NEEDLE_ANSWER.strip() in " ".join(top5_tokens)
        bounded_hits.append(hit)
        status = "HIT" if hit else "MISS"
        print(f"  depth={depth:5d}: {status}  top5={top5_tokens}")

    needle_results["configs"][f"coda_{config_name}"] = {
        "hits": bounded_hits,
        "accuracy": sum(bounded_hits) / len(bounded_hits),
        "config": config,
    }

    del bounded_model
    torch.cuda.empty_cache()

# Summary
print(f"\n{'='*60}")
print("NEEDLE RETRIEVAL SUMMARY")
print(f"{'='*60}")
print(f"{'Config':<25} | {'Accuracy':>8} | Depths: {DEPTHS}")
print("-" * 60)
for name, data in needle_results["configs"].items():
    hits_str = "".join(["Y" if h else "N" for h in data["hits"]])
    print(f"{name:<25} | {data['accuracy']*100:>6.1f}% | {hits_str}")

save_result(needle_results, "needle_retrieval")

---
## 5. H100 Throughput Benchmarks

Re-run the benchmark suite on H100 for paper-grade numbers.

In [ ]:
# Run the full benchmark suite
!python ../benchmarks/run_suite.py --prefill-lengths 512,2048,4096,8192

# Render tables
!python ../benchmarks/render_tables.py

In [ ]:
# Display the rendered tables
tables_path = RESULTS_DIR / "tables.md"
if tables_path.exists():
    from IPython.display import Markdown, display
    display(Markdown(tables_path.read_text()))

---
## 6. SDPA Backend Verification

Verify that the dense packing fix actually selects FlashAttention.

In [ ]:
# =====================================================================
# Verify SDPA backend selection
# =====================================================================

from coda_gqa_l import CoDAGQALandmarkPerf2

# Create a bounded model
test_model = CoDAGQALandmarkPerf2(
    embed_dim=576, num_heads=9, num_kv_heads=3,
    window=256, num_landmarks_exact=64, num_landmarks_summary=64,
).to(device=device, dtype=dtype)
test_model.eval()

state = test_model.init_state(batch_size=1, device=device, dtype=dtype)
x = torch.randn(1, 512, 576, device=device, dtype=dtype)

# Check with profiler which SDPA backend is used
with torch.no_grad():
    # Warm up
    _, state = test_model.prefill_chunked(x, state, block_size=256, write_cache=True, return_outputs=False)

    # Profile
    state2 = test_model.init_state(batch_size=1, device=device, dtype=dtype)
    with torch.autograd.profiler.profile(use_cuda=True) as prof:
        _, _ = test_model.prefill_chunked(x, state2, block_size=256, write_cache=True, return_outputs=False)

    # Look for SDPA kernel names
    sdpa_events = [e for e in prof.key_averages() if "sdpa" in e.key.lower()
                   or "flash" in e.key.lower()
                   or "efficient" in e.key.lower()
                   or "attention" in e.key.lower()]

    print("SDPA-related profiler events:")
    for e in sdpa_events:
        print(f"  {e.key}: {e.cuda_time_total/1e3:.2f} ms ({e.count} calls)")

    if not sdpa_events:
        print("  (No SDPA-specific events found, checking all CUDA kernels...)")
        for e in sorted(prof.key_averages(), key=lambda x: -x.cuda_time_total)[:15]:
            print(f"  {e.key}: {e.cuda_time_total/1e3:.2f} ms ({e.count} calls)")

del test_model, state, state2
torch.cuda.empty_cache()

---
## 7. V-Routing vs K-Routing Ablation

Demonstrates why V-routing matters: same token at different positions
should have high similarity in V-space but low in K-space (due to RoPE).

In [ ]:
# =====================================================================
# V-routing vs K-routing: position-invariance demonstration
# =====================================================================

test_model = CoDAGQALandmarkPerf2(
    embed_dim=576, num_heads=9, num_kv_heads=3,
    window=64, num_landmarks_exact=32, num_landmarks_summary=32,
    collect_metrics=True,
).to(device=device, dtype=dtype)
test_model.eval()

# Create input with repeated tokens at different positions
# E.g., same embedding vector at positions 10, 100, 200, 300
torch.manual_seed(42)
B, L, D = 1, 400, 576
x = torch.randn(B, L, D, device=device, dtype=dtype)

# Make tokens at positions 10, 100, 200, 300 identical
reference_token = x[:, 10:11, :].clone()
for pos in [100, 200, 300]:
    x[:, pos:pos+1, :] = reference_token

# Process through bounded attention
state = test_model.init_state(batch_size=1, device=device, dtype=dtype)
with torch.no_grad():
    _, state = test_model.prefill_chunked(
        x, state, block_size=64, write_cache=True, return_outputs=False,
    )

# Check metrics
print("Memory bank metrics after processing 400 tokens:")
print(f"  (4 identical tokens at positions 10, 100, 200, 300)")
print()
if state.metrics:
    for k, v in state.metrics.items():
        print(f"  {k}: {v}")

# Check V-space similarity of the identical tokens
# Project and check
with torch.no_grad():
    # Get V projections
    v_vecs = test_model.v_proj(reference_token)  # same input -> same V
    v_norm = F.normalize(v_vecs.view(1, 3, 1, 64), dim=-1)

    # Get K projections at different positions
    from coda_gqa_l.primitives import apply_rope
    k_raw = test_model.k_proj(reference_token).view(1, 3, 1, 64)

    positions = [10, 100, 200, 300]
    k_at_positions = []
    for pos in positions:
        cos, sin = test_model.rope(seq_len=1, offset=pos, device=device, dtype=dtype)
        k_roped = apply_rope(k_raw, cos, sin)
        k_at_positions.append(k_roped)

    # Cosine similarity in K-space (position-dependent due to RoPE)
    print("\nK-space cosine similarity (same token, different positions):")
    for i in range(len(positions)):
        for j in range(i+1, len(positions)):
            ki = F.normalize(k_at_positions[i], dim=-1)
            kj = F.normalize(k_at_positions[j], dim=-1)
            sim = (ki * kj).sum(dim=-1).mean().item()
            print(f"  pos {positions[i]} vs {positions[j]}: {sim:.4f}")

    # V-space similarity (always ~1.0 since V has no RoPE)
    print("\nV-space cosine similarity (same token, any position):")
    print(f"  Always: {1.0:.4f} (V is position-invariant, no RoPE)")

print("\n==> V-routing correctly identifies these as the same token,")
print("    while K-routing would treat them as different tokens.")

del test_model, state
torch.cuda.empty_cache()

---
## 8. Results Summary

Collate all results for the paper.

In [ ]:
# =====================================================================
# Final summary
# =====================================================================

print("=" * 70)
print("CoDA-GQA-L PAPER EXPERIMENT RESULTS")
print("=" * 70)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Model: {MODEL_NAME}")
print()

print("PERPLEXITY (WikiText-2):")
print(f"  Original SmolLM2-135M:     {ppl_original:.2f}")
print(f"  CoDA cold-swap (no train): {ppl_cold:.2f}")
print(f"  CoDA fine-tuned unbounded: {ppl_coda_unbounded:.2f}")
for name, ppl in bounded_results.items():
    print(f"  CoDA fine-tuned {name}: {ppl:.2f}")
print()

print("NEEDLE RETRIEVAL:")
for name, data in needle_results["configs"].items():
    print(f"  {name}: {data['accuracy']*100:.0f}% accuracy")
print()

print("MEMORY SAVINGS (at L=2048):")
for name, config in BOUNDED_CONFIGS.items():
    W, Me, Ms = config["window"], config["Me"], config["Ms"]
    bounded_kv = 2 * 3 * (W + Me + Ms) * 64 * 2  # 2 (k+v) * Hkv * Lbuf * Dh * sizeof(bf16)
    unbounded_kv = 2 * 3 * 2048 * 64 * 2
    ratio = unbounded_kv / bounded_kv
    print(f"  {name}: {bounded_kv/1024:.1f} KB ({ratio:.0f}x compression)")
print()

print(f"All results saved to {RESULTS_DIR}/")
print("=" * 70)

In [ ]:
# List all result files
for f in sorted(RESULTS_DIR.glob("*.json")):
    size = f.stat().st_size
    print(f"  {f.name} ({size/1024:.1f} KB)")